# 팀 라이브러리 에이전트 v2 — 설치 & 연결 노트북

**시작 전 체크리스트**
1. `config/.env.template` → `config/.env` 복사 후 키 입력
2. `LLM_PROVIDER` 를 `claude` / `chatgpt` / `gemini` 중 선택
3. Google Sheets 사용 시 `config/service_account.json` 추가


In [ ]:
!pip install -r ../requirements.txt -q

## 1. 환경변수 확인

In [1]:
import os, sys
sys.path.insert(0, "../scripts")
from dotenv import load_dotenv
load_dotenv("../config/.env")

provider = os.getenv('LLM_PROVIDER', 'claude')
print(f"현재 LLM 프로바이더: {provider.upper()}")
print()

keys = ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_API_KEY",
        "GOOGLE_SHEET_ID", "NOTION_TOKEN"]
for k in keys:
    status = '✅ 설정됨' if os.getenv(k) else '❌ 없음'
    print(f"{k}: {status}")

현재 LLM 프로바이더: CLAUDE

ANTHROPIC_API_KEY: ✅ 설정됨
OPENAI_API_KEY: ❌ 없음
GOOGLE_API_KEY: ❌ 없음
GOOGLE_SHEET_ID: ❌ 없음
NOTION_TOKEN: ✅ 설정됨


## 2. LLM 연결 테스트 (선택한 프로바이더 자동 적용)

In [2]:
import importlib, llm
importlib.reload(llm)  # .env 변경 후 재로드

resp = llm.chat("안녕! 지금 어떤 AI 모델이야? 한 문장으로 답해줘.")
print(f"[{llm.current_provider().upper()}] {resp}")

[CLAUDE] 저는 Anthropic에서 개발한 Claude라는 AI 어시스턴트입니다.


## 3. Notion 연결 테스트

In [3]:
import notion

token = os.getenv("NOTION_TOKEN")
if token:
    print(f"🔍 로드된 토큰 확인: {token[:10]}****************")
else:
    print("❌ 에러: .env 파일에서 NOTION_TOKEN을 찾을 수 없습니다. 경로가 맞는지 확인하세요.")
    

try:
    client = notion.get_client()
    me = client.users.me()
    print(f"✅ Notion 연결 성공! 사용자: {me.get('name')}")
except Exception as e:
    print(f"❌ 실패: {e}")

🔍 로드된 토큰 확인: ntn_635825****************
✅ Notion 연결 성공! 사용자: agent


## 4. API 모드 — Notion 문서 Q&A

In [8]:
# ---- 설정 ----
NOTION_PAGE_ID = "22cecdd06f97800d8500c7ffe511fe21"   # Notion 페이지 ID 입력
QUESTION = "이 문서의 핵심 내용을 3줄로 요약해줘."
# --------------

if not NOTION_PAGE_ID:
    print("NOTION_PAGE_ID 를 입력하세요.")
else:
    doc_text = notion.get_page_text(NOTION_PAGE_ID)
    answer = llm.chat(
        prompt=f"[문서]\n{doc_text}\n\n[질문]\n{QUESTION}",
        system="문서 내용만을 바탕으로 질문에 답하세요. 없는 내용은 모른다고 하세요."
    )
    print(answer)

문서 내용을 3줄로 요약하면 다음과 같습니다:

1. 이 페이지에는 2개의 연결된 데이터베이스가 있습니다.
2. 데이터베이스 탭을 클릭하면 같은 데이터를 다른 보기로 확인할 수 있습니다.
3. 프로젝트나 작업에 마우스 커서를 가져가고 ◨ 열기를 클릭하면 각각의 세부 정보를 볼 수 있습니다.


## 5. NotebookLM 모드 — Notion 페이지를 파일로 저장

In [10]:
# NotebookLM 에서는 API 없이 로컬 파일을 소스로 업로드합니다.
# 아래 셀로 Notion 페이지를 .txt 로 저장 후 NotebookLM 에 업로드하세요.

NOTION_PAGE_ID = "22cecdd06f97800d8500c7ffe511fe21"          # Notion 페이지 ID 입력
OUTPUT_PATH = "../output/notion_export.txt"   # 저장 경로

import os; os.makedirs("../output", exist_ok=True)

if not NOTION_PAGE_ID:
    print("NOTION_PAGE_ID 를 입력하세요.")
else:
    saved = notion.save_page_to_file(NOTION_PAGE_ID, OUTPUT_PATH)
    print(f"NotebookLM 에서 '{saved}' 파일을 '소스 추가 → 파일 업로드' 로 불러오세요.")

✅ 저장 완료: ../output/notion_export.txt
NotebookLM 에서 '../output/notion_export.txt' 파일을 '소스 추가 → 파일 업로드' 로 불러오세요.


## 6. 로컬 LLM 테스트 (LLM_PROVIDER=local 인 경우)

> ollama serve 가 실행 중이어야 합니다.

In [ ]:
# LLM_PROVIDER=local 일 때만 실행하세요.
# 먼저 터미널에서: ollama serve

import importlib, llm
importlib.reload(llm)

if llm.current_provider() == "local":
    try:
        resp = llm.chat("안녕하세요! 한 문장으로 인사해주세요.")
        print(f"✅ 로컬 LLM 연결 성공!
응답: {resp}")
    except Exception as e:
        print(f"❌ 연결 실패: {e}")
else:
    print(f"현재 프로바이더: {llm.current_provider()} (local 아님 — 스킵)")


## 7. STT 캘린더 에이전트

> **사전 준비**
> 1.  — Google Cloud 서비스 계정 키
> 2. 해당 서비스 계정 이메일을 Google Calendar 공유 설정에 **편집자**로 추가
> 3.  의  설정

녹취록 텍스트를 넣으면 **Test 1(일정 추출 → 캘린더 등록)**과 **Test 2(인사이트 요약)**를 동시에 실행합니다.

In [ ]:
import importlib, stt_calendar
importlib.reload(stt_calendar)

# ---- 캘린더 연동 설정 확인 ----
from dotenv import load_dotenv
load_dotenv("../config/calendar.env")

import os
sa = os.getenv("GOOGLE_SERVICE_ACCOUNT", "config/service_account.json")
cal_id = os.getenv("GOOGLE_CALENDAR_ID", "primary")
sa_ok = os.path.exists("../" + sa)

print(f"서비스 계정 파일: {sa}  {'✅' if sa_ok else '❌ 파일 없음'}")
print(f"캘린더 ID       : {cal_id}")

In [ ]:
# ---- 녹취록 텍스트 입력 ----
TRANSCRIPT = """
여기에 STT로 변환된 녹취록 텍스트를 붙여넣으세요.

예시)
지영: 다음 스터디 때 피그마 MCP 연동 테스트 해보자.
인선: 좋아, 내가 구글 시트 오류 원인 분석해서 주말까지 정리할게.
지영: 그리고 커서 활용법도 공유해주면 좋겠어.
"""

# skip_calendar=True 로 하면 캘린더 등록 없이 추출만 합니다
events, markdown = stt_calendar.run(
    transcript=TRANSCRIPT,
    skip_calendar=False,
    insights_path="../output/insights.md",
)

In [ ]:
# Test 1 결과만 별도 확인
import json
print("=== 추출된 일정 (JSON) ===")
print(json.dumps(events, ensure_ascii=False, indent=2))

In [ ]:
# Test 2 결과만 별도 확인
print("=== 인사이트 Markdown ===")
print(markdown)